# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ariba86/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
!pip install -q duckdb huggingface_hub


In [3]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Setup done!")

Setup done!


In [4]:
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

con.sql(f"""
    CREATE OR REPLACE VIEW march_data AS
    SELECT * FROM read_parquet('{path}')
""")

print("View ready!")

View ready!


In [5]:
signal1 = con.sql("""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1_top3'
            WHEN gsc_avg_position <= 10 THEN '2_top10'
            WHEN gsc_avg_position <= 20 THEN '3_top20'
            ELSE '4_below20'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr
    FROM march_data
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY position_bucket
    ORDER BY position_bucket
""")
signal1.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬───────────────────────┐
│ position_bucket │    n    │        avg_ctr        │
│     varchar     │  int64  │        double         │
├─────────────────┼─────────┼───────────────────────┤
│ 1_top3          │  727362 │  0.004755515699920511 │
│ 2_top10         │ 1456122 │ 0.0034726355196117438 │
│ 3_top20         │  519223 │  0.002769909681133759 │
│ 4_below20       │  908354 │ 0.0012891531178192733 │
└─────────────────┴─────────┴───────────────────────┘



In [6]:
signal2 = con.sql("""
    SELECT
        CASE
            WHEN gsc_impressions < 10 THEN '1_low_vol'
            WHEN gsc_impressions < 100 THEN '2_medium_vol'
            WHEN gsc_impressions < 1000 THEN '3_high_vol'
            ELSE '4_very_high_vol'
        END AS volume_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr,
        SUM(gsc_clicks) AS total_clicks
    FROM march_data
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY volume_bucket
    ORDER BY volume_bucket
""")
signal2.show()

┌─────────────────┬─────────┬───────────────────────┬──────────────┐
│  volume_bucket  │    n    │        avg_ctr        │ total_clicks │
│     varchar     │  int64  │        double         │    int128    │
├─────────────────┼─────────┼───────────────────────┼──────────────┤
│ 1_low_vol       │ 1463532 │ 0.0034905942805292167 │        14874 │
│ 2_medium_vol    │ 1508921 │ 0.0026945079741282304 │       157424 │
│ 3_high_vol      │  606189 │ 0.0030722660344846713 │       485209 │
│ 4_very_high_vol │   32419 │ 0.0027144768063721947 │       164325 │
└─────────────────┴─────────┴───────────────────────┴──────────────┘



**Signal 1: Position → CTR (flag-linked: behind the CTR-fix logic)**

Bucketed gsc_avg_position into top3 / top10 / top20 / below20 and measured
average CTR per bucket (n = 727,362 / 1,456,122 / 519,223 / 908,354).

Result: CTR falls steadily as position gets worse — top3 = 0.00476,
top10 = 0.00347, top20 = 0.00277, below20 = 0.00129.

**Verdict: CONFIRMED** — better position genuinely means better CTR,
so a page ranking well but underperforming on CTR is a real, checkable signal.

---

**Signal 2: Volume/Impressions (flag-linked: behind quick-win logic)**

Bucketed gsc_impressions into low / medium / high / very-high volume and
measured avg CTR and total clicks per bucket (n = 1,463,532 / 1,508,921 /
606,189 / 32,419).

Result: avg_ctr stays roughly flat across volume buckets (0.0027-0.0035) —
volume does NOT predict CTR. But total_clicks is highly concentrated in the
high-volume bucket (485,209 clicks from only 606,189 rows), meaning a fix on
a high-volume page has far more real-world impact than the same fix on a
low-volume page.

**Verdict: MIXED** — volume doesn't change CTR itself, but it changes how much
a CTR fix is worth. Used as an impact multiplier, not a CTR predictor.

---

**My rule (plain words):**

Flag a page as a "CTR Fix" opportunity if it ranks reasonably well (position
≤ 20, so it's already earning search visibility) but its CTR is clearly below
what pages at that position typically get. Rank these by impressions, so pages
with the most visibility (and therefore the most potential clicks to gain) are
reviewed first.

**Reason codes this rule can output:**
- `CTR_BELOW_POSITION_EXPECTED` — page ranks fine but underperforms on CTR
  for its position bucket
- `NO_FLAG` — page's CTR is at or above what's expected for its position

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Step 1: aggregate each page (client+content) to March-month level
page_summary = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM march_data
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

page_summary['actual_ctr'] = page_summary['total_clicks'] / page_summary['total_impressions']

# Step 2: assign each page to a position bucket + expected CTR (from Signal 1 findings)
def position_bucket(pos):
    if pos <= 3: return '1_top3'
    elif pos <= 10: return '2_top10'
    elif pos <= 20: return '3_top20'
    else: return '4_below20'

expected_ctr_map = {
    '1_top3': 0.00476,
    '2_top10': 0.00347,
    '3_top20': 0.00277,
    '4_below20': 0.00129
}

page_summary['position_bucket'] = page_summary['avg_position'].apply(position_bucket)
page_summary['expected_ctr'] = page_summary['position_bucket'].map(expected_ctr_map)

# Step 3: the score = how far below expected CTR is, weighted by impressions (impact)
page_summary['ctr_gap'] = page_summary['expected_ctr'] - page_summary['actual_ctr']
page_summary['score'] = page_summary['ctr_gap'] * page_summary['total_impressions']
page_summary['score'] = page_summary['score'].clip(lower=0)  # only positive gaps matter

# Step 4: reason code + action label
page_summary['reason_code'] = page_summary.apply(
    lambda r: 'CTR_BELOW_POSITION_EXPECTED' if r['ctr_gap'] > 0 else 'NO_FLAG', axis=1
)
page_summary['action'] = page_summary['reason_code'].apply(
    lambda r: 'review_title_and_meta' if r == 'CTR_BELOW_POSITION_EXPECTED' else 'no_action'
)

# Step 5: rank and save
ranked_queue = page_summary.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Saved! Shape:", ranked_queue.shape)
ranked_queue.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved! Shape: (176738, 12)


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,actual_ctr,position_bucket,expected_ctr,ctr_gap,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,2_top10,0.00347,0.003357,713.04188,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,1_top3,0.00476,0.003340,679.64572,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,2_top10,0.00347,0.003463,467.39448,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,2_top10,0.00347,0.003169,453.27593,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,2_top10,0.00347,0.003462,429.54025,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,5.789019,0.000626,2_top10,0.00347,0.002844,377.09771,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,2_top10,0.00347,0.003331,358.31648,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
7,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,1_top3,0.00476,0.004327,349.70796,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
8,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,1_top3,0.00476,0.001507,333.43560,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
9,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,2.481596,0.001031,1_top3,0.00476,0.003729,332.73004,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top20 = ranked_queue.head(20)[['client_hash_id', 'content_hash_id', 'total_impressions',
                                  'total_clicks', 'avg_position', 'actual_ctr',
                                  'expected_ctr', 'score', 'reason_code', 'action']]
top20


,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,actual_ctr,expected_ctr,score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,0.00347,713.04188,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.563756,0.001420,0.00476,679.64572,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,0.00347,467.39448,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
3,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,0.00347,453.27593,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
4,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,0.00347,429.54025,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,5.789019,0.000626,0.00347,377.09771,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,0.00347,358.31648,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
7,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,1.488604,0.000433,0.00476,349.70796,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
8,client_e547b89c05043229,content_0e03de7680314cd5,221310.0,720.0,2.675217,0.003253,0.00476,333.43560,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta
9,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,2.481596,0.001031,0.00476,332.73004,CTR_BELOW_POSITION_EXPECTED,review_title_and_meta


**Top-20 Review**

1. content_44f34c0a90047651 — action: review_title_and_meta. High impressions
   (212,404) with near-zero CTR (0.0113%) at position ~7. What would make this
   wrong: if this page's queries are mostly "navigational" (brand searches),
   users may already know the URL and skip clicking from search.

2. content_8d7d99f109e19aa2 — action: review_title_and_meta. Strong volume
   (203,497) at top3 position but CTR far below expected. What would make this
   wrong: if a rich snippet/featured answer is satisfying the query without a click.

3. content_8e1334d6356668e3 — action: review_title_and_meta. Huge impressions
   (134,984) but only 1 total click all month. What would make this wrong: if
   impressions are inflated by bot/crawler traffic rather than real users.

4. content_34a70fea29d15f24 — action: review_title_and_meta. 143,019 impressions,
   43 clicks, position ~3. What would make this wrong: if the query intent is
   informational and the snippet already answers it (no need to click through).

5. content_fec55986a1868d62 — action: review_title_and_meta. 124,075 impressions,
   1 click only. What would make this wrong: same bot-traffic risk as row 3 —
   worth checking impression source before assuming a real title/meta problem.

6. content_7c8373141eae744a — action: review_title_and_meta. Decent volume
   (132,593), CTR far under expected for position ~6. What would make this wrong:
   if this page recently changed URL/content and GSC data is still catching up.

7. content_f6116743b00afc2d — action: review_title_and_meta. 107,584 impressions,
   15 clicks. What would make this wrong: if title/meta were already recently
   updated and this data predates the fix — the flag would be stale.

8. content_306bc78dff1eb683 — action: review_title_and_meta. Position ~1.5
   (excellent) but CTR still under expected. What would make this wrong: if
   expected_ctr for top3 is too high for this specific query type (e.g.
   image-heavy SERPs reduce clicks even at position 1).

9. content_0e03de7680314cd5 — action: review_title_and_meta. Highest total_clicks
   in the top 20 (720) but still flagged — meaningful absolute opportunity.
   What would make this wrong: unlikely to be wrong; this one has real signal —
   good candidate to prioritize first.

10. content_9ef3d7516483e665 — action: review_title_and_meta. Position ~2.5,
    92 clicks, CTR still below expected. What would make this wrong: if seasonal
    demand for this query dropped in March specifically (need to check other months).

11. content_acbcc847f8996314 — action: review_title_and_meta. 170,808 impressions,
    262 clicks. What would make this wrong: unlikely — decent click volume already,
    but still under the position-expected benchmark, a fair CTR-fix candidate.

12. content_b99ea6881864dea5 — action: review_title_and_meta. 194,337 impressions,
    361 clicks. What would make this wrong: if this page targets a broad set of
    queries where average position hides real ranking variance across keywords.

13. content_cd3d932d4e1c8db0 — action: review_title_and_meta. Only 4 total clicks
    from 89,332 impressions. What would make this wrong: strong bot-traffic
    suspicion again — very low absolute clicks is a red flag for data quality,
    not necessarily a content problem.

14. content_4ffe18112a5642e3 — action: review_title_and_meta. 186,983 impressions,
    586 clicks — healthy click count already. What would make this wrong:
    marginal gain here may be small since CTR gap, while flagged, isn't huge.

15. content_545bb6cc7081ded3 — action: review

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Check: how many top-20 picks have suspiciously low absolute clicks (possible bot traffic, not real signal)
weak_check = ranked_queue.head(20).copy()
weak_check['low_click_flag'] = weak_check['total_clicks'] < 20

print("Weak picks (total_clicks < 20) in top 20:", weak_check['low_click_flag'].sum())
weak_check[weak_check['low_click_flag']][['content_hash_id', 'total_impressions', 'total_clicks', 'score']]


Weak picks (total_clicks < 20) in top 20: 7


,content_hash_id,total_impressions,total_clicks,score
2,content_8e1334d6356668e3,134984.0,1.0,467.39448
4,content_fec55986a1868d62,124075.0,1.0,429.54025
6,content_f6116743b00afc2d,107584.0,15.0,358.31648
12,content_cd3d932d4e1c8db0,89332.0,4.0,305.98204
17,content_046fc480045b88f5,83788.0,6.0,284.74436
18,content_9540d884af3e41fd,82376.0,11.0,274.84472
19,content_fc67675904376267,60172.0,18.0,268.41872


**Weak picks (7 of top 20 flagged as shaky):**

7 of my top-20 picks have very low absolute clicks (<20 total clicks for the
month) despite high impressions — e.g. content_8e1334d6356668e3 (134,984
impressions, only 1 click) and content_cd3d932d4e1c8db0 (89,332 impressions,
only 4 clicks).

These are flagged by the score formula because a huge impression count times
a tiny CTR still produces a large "gap × impressions" score — but 1-4 clicks
out of 100k+ impressions is more consistent with bot/crawler traffic inflating
impressions than with a genuine title/meta problem. Before acting on these,
I'd want to check click patterns over multiple months, not just March alone,
to rule out a data-quality issue rather than a real content issue.

The stronger picks (like content_0e03de7680314cd5 with 720 real clicks, or
content_b99ea6881864dea5 with 361 clicks) have enough actual click volume to
trust the CTR gap as a real signal.

**Leakage check:**

- No future dates were used: all features (total_impressions, total_clicks,
  avg_position, actual_ctr) are computed purely from March 2026 history —
  the same month being scored, with no next-day or next-month values folded in.
- expected_ctr came from Signal 1's bucket averages, computed on the full
  March dataset — not from any single page's own future outcome.
- No product/UI flags (like an existing "needs refresh" tag) were used as
  inputs — the score is built entirely from position and CTR, both directly
  observed in this same window.
- This is decision-support only: it ranks candidates for human review, not
  an automated action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.